# RAASTA — Fine-tune for people + animals

**Goal:** Make `person` and animal classes (cow, buffalo, dog, cat, horse, donkey, goat) more reliable, while keeping road damage (pothole / crack / speed_bump).

## Critical rule
Keep the **same 11 classes**. Do **not** create separate classes for child / man / woman / old.

| Real life | Train as | Why |
|-----------|----------|-----|
| Child, man, woman, elderly | **`person`** (class 3) | Phone app only has `person` |
| Cow breeds, calves | **`cow`** | Diversity inside one class |
| Donkey vs horse | Classes 8 / 9 | More donkey images so horse stops stealing them |

## Before you run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Google Drive with ~5 GB free (`MyDrive/raasta/`)
3. Free [Roboflow](https://roboflow.com) account → copy **API key**
4. Existing weights somewhere under:
   - `MyDrive/raasta/weights/m2_m5_ft1/finetune/weights/best.pt` **or**
   - `MyDrive/raasta/weights/m2_m5_v1/weights/best.pt`
5. Existing merged dataset: `MyDrive/raasta/merged/data.yaml`

**Time:** ~2–4 hours on T4 (50 epochs). Run cells **top to bottom**.

## 0) Check GPU
You must see `True` and a GPU name. If not: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 1) Setup + Drive
Mount Drive, install packages, find your latest `best.pt`.

In [ ]:
#@title 1) Install + mount Drive + find best.pt
from google.colab import drive
drive.mount("/content/drive")

!pip -q uninstall -y pillow pillow-simd 2>/dev/null
!pip -q install -U "pillow==11.2.1" "ultralytics>=8.3.0" roboflow pyyaml

import os, shutil, random, yaml, gc
from pathlib import Path
from collections import Counter, defaultdict
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

DRIVE = Path("/content/drive/MyDrive/raasta")
MERGED = DRIVE / "merged"

# Prefer latest fine-tune if it exists; else original best
CANDIDATES = [
    DRIVE / "weights/m2_m5_ft1/finetune/weights/best.pt",
    DRIVE / "weights/m2_m5_v1/weights/best.pt",
]
BEST = next((p for p in CANDIDATES if p.exists()), None)
assert BEST is not None, f"No best.pt found under {DRIVE}/weights — upload your weights first"
assert (MERGED / "data.yaml").exists(), f"Missing {MERGED}/data.yaml — need your previous merged dataset"
print("Starting weights:", BEST)
print("Merged data:", MERGED)

## 2) Fixed 11-class map (must match Flutter)
Child / man / woman / elderly → all map to **`person`**.

In [ ]:
#@title 2) Class names + aliases
CLASS_NAMES = [
    "pothole", "crack", "speed_bump",
    "person", "cow", "buffalo", "dog", "cat", "horse", "donkey", "goat",
]
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
print(CLASS_TO_ID)

ALIASES = {
    # person family → class person
    "person": "person", "pedestrian": "person", "people": "person",
    "human": "person", "man": "person", "woman": "person",
    "child": "person", "boy": "person", "girl": "person",
    "kid": "person", "elderly": "person", "old": "person",
    "adult": "person", "walker": "person",
    # animals
    "cow": "cow", "cattle": "cow", "ox": "cow", "bull": "cow",
    "buffalo": "buffalo", "water buffalo": "buffalo",
    "dog": "dog", "puppy": "dog",
    "cat": "cat", "kitten": "cat",
    "horse": "horse",
    "donkey": "donkey", "ass": "donkey", "mule": "donkey",
    "goat": "goat",
    # road (keep skill)
    "pothole": "pothole", "crack": "crack", "speed_bump": "speed_bump",
    "speed bump": "speed_bump", "speedbreaker": "speed_bump", "bump": "speed_bump",
}

def canon(name: str):
    k = name.strip().lower().replace("_", " ").replace("-", " ")
    return ALIASES.get(k)

print("Aliases ready. person examples:", [k for k, v in ALIASES.items() if v == "person"])

## 3) Download extra datasets (Roboflow)
Paste your API key in the cell below.

If one project 404s, that line is skipped automatically — you still get the others.

**Tip:** On [Roboflow Universe](https://universe.roboflow.com) search `pedestrian`, `donkey detection`, `buffalo`, `goat` — then add more `rf_download(...)` lines if needed.

In [ ]:
#@title 3) Roboflow downloads
ROBOFLOW_API_KEY = ""  #@param {type:"string"}
assert ROBOFLOW_API_KEY.strip(), "Paste Roboflow API key above"

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY.strip())

RAW = Path("/content/ft_raw")
shutil.rmtree(RAW, ignore_errors=True)
RAW.mkdir(parents=True)

def rf_download(workspace, project, version, folder):
    out = RAW / folder
    print(f"Downloading {workspace}/{project} v{version} …")
    ds = rf.workspace(workspace).project(project).version(version).download(
        "yolov8", location=str(out)
    )
    print(" →", ds.location)
    return Path(ds.location)

downloads = []

# --- PEOPLE (map everything to person later) ---
try:
    downloads.append(("person", rf_download("roboflow-100", "people-in-paintings", 1, "rf_people_a")))
except Exception as e:
    print("skip people-a:", e)

try:
    downloads.append(("person", rf_download("public-datasets", "coco-person", 1, "rf_coco_person")))
except Exception as e:
    print("skip coco-person:", e)
    try:
        downloads.append(("person", rf_download("microsoft", "coco", 1, "rf_coco")))
    except Exception as e2:
        print("skip coco:", e2)

# --- ANIMALS (boost weak classes: donkey, goat, buffalo especially) ---
animal_jobs = [
    ("cow", "joseph-nelson", "cows", 1, "rf_cows"),
    ("dog", "joseph-nelson", "dogs", 1, "rf_dogs"),
    ("cat", "joseph-nelson", "cats", 1, "rf_cats"),
    ("horse", "joseph-nelson", "horses", 1, "rf_horses"),
    ("goat", "augmented-startups", "goats-clyzb", 1, "rf_goats"),
    ("donkey", "search", "donkey-detection", 1, "rf_donkey"),
    ("buffalo", "search", "buffalo-detection", 1, "rf_buffalo"),
]

for canon_name, ws, proj, ver, folder in animal_jobs:
    try:
        downloads.append((canon_name, rf_download(ws, proj, ver, folder)))
    except Exception as e:
        print(f"skip {canon_name} ({ws}/{proj}):", e)

print("Downloaded sets:", len(downloads))
for c, p in downloads:
    print(" ", c, "→", p)

assert len(downloads) >= 1, "No datasets downloaded — check API key / swap Universe projects"

## 4) Build balanced fine-tune mix (FAST version)

**Do not wait on the old slow cell** — stop it (■) and run this instead.

Quality for the app is the same idea: keep road skill from `merged` + boost person/animals. We use ~2000 old train images (not 4000) and parallel copies so Drive I/O finishes in minutes, not half an hour.

**Healthy targets:** high `person`; each animal a few hundred boxes; road classes from old sample.

In [ ]:
#@title 4) Build /content/ft_person_animals (FAST — use this)
# Stop any old Step-4 cell still running, then run this.
from concurrent.futures import ThreadPoolExecutor, as_completed

OUT = Path("/content/ft_person_animals")
shutil.rmtree(OUT, ignore_errors=True)
for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)

rng = random.Random(42)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def yolo_pairs(root: Path):
    pairs = []
    candidates = [
        (root / "train" / "images", root / "train" / "labels"),
        (root / "valid" / "images", root / "valid" / "labels"),
        (root / "val" / "images", root / "val" / "labels"),
        (root / "images" / "train", root / "labels" / "train"),
        (root / "images" / "val", root / "labels" / "val"),
        (root / "images" / "valid", root / "labels" / "valid"),
    ]
    for img_dir, lbl_dir in candidates:
        if not img_dir.exists() or not lbl_dir.exists():
            continue
        for img in img_dir.iterdir():
            if not img.is_file() or img.suffix.lower() not in IMG_EXT:
                continue
            lbl = lbl_dir / f"{img.stem}.txt"
            if lbl.exists() and lbl.stat().st_size > 0:
                pairs.append((img, lbl))
    return pairs

def remap_label_file(src_lbl: Path, name_by_id, force_class):
    lines_out = []
    for line in src_lbl.read_text(encoding="utf-8", errors="ignore").strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        if force_class is not None:
            cid = CLASS_TO_ID[force_class]
        else:
            old_id = int(float(parts[0]))
            raw = name_by_id.get(old_id, str(old_id)) if name_by_id else str(old_id)
            cname = canon(raw)
            if cname is None:
                continue
            cid = CLASS_TO_ID[cname]
        parts[0] = str(cid)
        lines_out.append(" ".join(parts))
    return lines_out

def read_names(yaml_path: Path):
    if not yaml_path.exists():
        return None
    data = yaml.safe_load(yaml_path.read_text())
    names = data.get("names")
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    if isinstance(names, list):
        return {i: str(n) for i, n in enumerate(names)}
    return None

def fast_copy(src: Path, dst: Path):
    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)

copied = Counter()
n_files = {"train": 0, "val": 0}

def add_pair(img: Path, lbl_lines, split: str, prefix: str):
    if not lbl_lines:
        return
    stem = f"{prefix}_{img.stem}_{n_files[split]}"
    fast_copy(img, OUT / "images" / split / f"{stem}{img.suffix.lower()}")
    (OUT / "labels" / split / f"{stem}.txt").write_text("\n".join(lbl_lines) + "\n")
    n_files[split] += 1
    for line in lbl_lines:
        copied[CLASS_NAMES[int(line.split()[0])]] += 1

# A) Old merged — 2000/300 is enough for app quality (4000 was only slower)
def process_old(img: Path, lbl_dir: Path, split: str):
    lbl = lbl_dir / f"{img.stem}.txt"
    if not lbl.exists():
        return None
    lines = lbl.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    clean = []
    for line in lines:
        parts = line.split()
        if len(parts) >= 5 and int(float(parts[0])) < len(CLASS_NAMES):
            clean.append(" ".join(parts))
    return (img, clean, split)

for split, limit in [("train", 2000), ("val", 300)]:
    img_dir = MERGED / "images" / split
    lbl_dir = MERGED / "labels" / split
    print(f"Listing Drive merged/{split} …")
    imgs = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT]
    rng.shuffle(imgs)
    imgs = imgs[:limit]
    print(f"Copying {len(imgs)} old {split} images (parallel) …")
    jobs = []
    with ThreadPoolExecutor(max_workers=8) as ex:
        futs = [ex.submit(process_old, img, lbl_dir, split) for img in imgs]
        for i, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            if res:
                jobs.append(res)
            if i % 200 == 0 or i == len(futs):
                print(f"  read {i}/{len(futs)}")
    for img, clean, sp in jobs:
        add_pair(img, clean, sp, "old")

print("After old merged sample:", dict(copied), "files", n_files)

# B) New downloads (local /content — fast)
PER_SET_LIMIT = {"train": 900, "val": 120}
for force_or_auto, root in downloads:
    names = read_names(root / "data.yaml")
    pairs = yolo_pairs(root)
    print(f"{force_or_auto}: {len(pairs)} pairs from {root.name}")
    rng.shuffle(pairs)
    cut = int(len(pairs) * 0.85)
    buckets = {"train": pairs[:cut], "val": pairs[cut:]}
    for split, items in buckets.items():
        for img, lbl in items[: PER_SET_LIMIT[split]]:
            force = force_or_auto if force_or_auto in CLASS_TO_ID else None
            lines = remap_label_file(lbl, names, force)
            add_pair(img, lines, split, force_or_auto or "mix")

print("Final box counts:", dict(copied))
print("Files:", n_files)
yaml_path = OUT / "data.yaml"
yaml_path.write_text(yaml.safe_dump({
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}, sort_keys=False))
print("Wrote", yaml_path)
print("DONE — go to Step 5")
if copied.get("donkey", 0) < 50:
    print("WARNING: donkey still low")
if copied.get("person", 0) < 200:
    print("WARNING: person still low")

## 5) Train (fine-tune)
Starts from your existing `best.pt` with a **low learning rate**.

- If **CUDA out of memory** → change `batch=8` to `batch=4`
- If Colab disconnects mid-train → run the **resume** cell below

In [ ]:
#@title 5) Fine-tune 50 epochs
gc.collect()
torch.cuda.empty_cache()

RUN = DRIVE / "weights/m2_m5_people_animals_v1"
RUN.mkdir(parents=True, exist_ok=True)

model = YOLO(str(BEST))
model.train(
    data=str(OUT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=8,          # if CUDA OOM → 4
    workers=2,
    device=0,
    project=str(RUN),
    name="finetune",
    exist_ok=True,
    pretrained=True,
    lr0=0.0006,       # low LR = fine-tune
    lrf=0.01,
    patience=15,
    cache=False,
    amp=True,
    plots=True,
    save_period=5,
    mosaic=0.4,
    close_mosaic=10,
    fliplr=0.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

FT_BEST = RUN / "finetune" / "weights" / "best.pt"
print("Fine-tuned best:", FT_BEST)

### Resume only (if Colab disconnected)
Skip this if training finished normally.

In [ ]:
#@title Resume interrupted training (optional)
# from ultralytics import YOLO
# last = DRIVE / "weights/m2_m5_people_animals_v1/finetune/weights/last.pt"
# YOLO(str(last)).train(resume=True)

### Validate + note metrics for FYP report
Realistic targets: overall mAP50 ~0.55–0.75; clear gain on `person`; donkey/goat better than before.

In [ ]:
#@title 5b) Validate best.pt
from ultralytics import YOLO

FT_BEST = DRIVE / "weights/m2_m5_people_animals_v1/finetune/weights/best.pt"
m = YOLO(str(FT_BEST))
metrics = m.val(data=str(OUT / "data.yaml"))
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
try:
    print("Per-class mAP50-95:", metrics.box.maps)
except Exception:
    pass

## 6) Export TFLite for the phone
Copies `.pt` + `.tflite` to `MyDrive/raasta/exports/`.

In [ ]:
#@title 6) Export TFLite + copy to Drive
from ultralytics import YOLO
import shutil

FT_BEST = DRIVE / "weights/m2_m5_people_animals_v1/finetune/weights/best.pt"
model = YOLO(str(FT_BEST))

export_path = model.export(format="tflite", imgsz=640)
print("TFLite:", export_path)

out = DRIVE / "exports"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(FT_BEST, out / "raasta_m2_m5_people_animals.pt")
shutil.copy2(export_path, out / "raasta_m2_m5_people_animals.tflite")
print("Copied to", out)
print("Download:", out / "raasta_m2_m5_people_animals.tflite")

## 7) Put the model in the Flutter app (on your PC)

1. Download from Drive:  
   `MyDrive/raasta/exports/raasta_m2_m5_people_animals.tflite`
2. Replace:  
   `E:\\Android\\projects\\raasta_app\\assets\\models\\raasta_m2_m5.tflite`
3. Run:

```powershell
cd E:\\Android\\projects\\raasta_app
flutter run -d YOUR_DEVICE_ID --device-timeout 60
```

**Do not change** Flutter `numClasses = 11` — this fine-tune keeps class IDs identical.

---

## What NOT to do
- Do not add classes `child`, `man`, `woman` without changing Flutter
- Do not train on CPU
- Do not start from `yolov8n.pt` for this pass — continue from `best.pt`
- Do not drop all old pothole data

When done, tell me your **mAP50** and which classes still fail on the phone.